In [0]:
from pyspark.sql import functions as F
updates_path = "/Volumes/workspace/default/book_data/first_project_orders_updates.csv"
updates_df = (
    spark.read.option("header", "true").csv(updates_path)
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("loaded_at", F.current_timestamp())
)


updates_df.createOrReplaceTempView("updates")

print("updates rows:", updates_df.count())

updates rows: 12


In [0]:
%sql
MERGE INTO workspace.first_project.orders_silver AS target
USING updates AS source
  ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,8,0,4
